# DJF ITCZ test with the Liao centroid method

This notebook is a first diagnostic test of the **Liao et al. (2023) centroid ITCZ latitude** using **DJF rainfall climatology only** over the Maritime Continent / Indo-Pacific sector.

Method used here:

- Rainfall source: the existing project MSWEP monthly workflow.
- Native source units are **mm month$^{-1}$** and this notebook keeps them in those units.
- The DJF climatology is built from all **complete DJF seasons in 1981-2020**.
- The centroid is computed in three latitude-band configurations:
  - NH ITCZ: **0 to 20N**
  - SH ITCZ: **20S to 0**
  - Donohoe-style equatorial centroid: **20S to 20N**
- For each longitude, the ITCZ latitude is the precipitation-weighted centroid

$$
\phi_{\mathrm{ITCZ}} = \frac{\int \phi \, P(\phi) \, \cos(\phi) \, d\phi}{\int P(\phi) \, \cos(\phi) \, d\phi}
$$

- The new equatorial line is labeled **-20, 20 Centroid**.
- A light Gaussian smoothing test with **sigma = 1 grid cell** is included only to check whether the climatological ITCZ path looks less patchy after conservative spatial smoothing.

This notebook does **not** compute anomalies, trends, or ENSO correlations.

In [ ]:
from data_processing.config import RAINFALL_PATH

from __future__ import annotations

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

MPLCONFIGDIR = Path('/tmp/matplotlib')
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from scipy.ndimage import gaussian_filter

warnings.filterwarnings(
    'ignore',
    message='invalid value encountered in create_collection',
    category=RuntimeWarning,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

In [ ]:
MSWEP_PATH = Path(rainfall_path = RAINFALL_PATH)

START_YEAR = 1981
END_YEAR = 2020
DJF_MONTHS = [12, 1, 2]
PLOT_EXTENT = [90.0, 170.0, -20.0, 20.0]
NH_BAND = (0.0, 20.0)
SH_BAND = (-20.0, 0.0)
FULL_BAND = (-20.0, 20.0)
SMOOTH_SIGMA = 1.0
RAIN_CONTOUR_LEVEL = 150.0

In [ ]:
def compute_djf_climatology(precip: xr.DataArray, start_year: int = 1981, end_year: int = 2020) -> tuple[xr.DataArray, np.ndarray]:
    """Build the mean DJF rainfall climatology from complete DJF seasons only."""
    djf = precip.sel(time=precip.time.dt.month.isin(DJF_MONTHS)).copy()
    time_index = pd.DatetimeIndex(djf['time'].values)
    djf_year = np.where(time_index.month == 12, time_index.year + 1, time_index.year).astype(int)
    djf = djf.assign_coords(djf_year=('time', djf_year))

    season_counts = pd.Series(djf_year).value_counts().sort_index()
    complete_years = season_counts[season_counts == 3].index.to_numpy(dtype=int)
    complete_years = complete_years[(complete_years >= start_year) & (complete_years <= end_year)]
    if complete_years.size == 0:
        raise ValueError('No complete DJF seasons were found for the requested period.')

    climatology = djf.sel(time=djf.djf_year.isin(complete_years)).mean('time', skipna=True)
    climatology.attrs = dict(precip.attrs)
    climatology.attrs['climatology_years'] = f'{int(complete_years.min())}-{int(complete_years.max())}'
    return climatology, complete_years


def smooth_precip_gaussian(precip: xr.DataArray, sigma: float = 1.0, lat_name: str = 'lat', lon_name: str = 'lon') -> xr.DataArray:
    """Apply conservative Gaussian smoothing while preserving missing values safely."""
    field = precip.transpose(lat_name, lon_name)
    values = np.asarray(field.values, dtype=float)
    valid = np.isfinite(values).astype(float)
    filled = np.where(np.isfinite(values), values, 0.0)

    smooth_values = gaussian_filter(filled, sigma=sigma, mode='nearest')
    smooth_valid = gaussian_filter(valid, sigma=sigma, mode='nearest')
    result = np.where(smooth_valid > 1e-6, smooth_values / smooth_valid, np.nan)

    smoothed = xr.DataArray(
        result,
        coords=field.coords,
        dims=field.dims,
        name=precip.name,
        attrs=dict(precip.attrs),
    )
    smoothed.attrs['smoothing'] = f'Gaussian sigma={sigma} grid cells'
    return smoothed.transpose(*precip.dims)


def compute_itcz_centroid(
    precip: xr.DataArray,
    lat_name: str = 'lat',
    lon_name: str = 'lon',
    lat_min: float = 0.0,
    lat_max: float = 20.0,
) -> tuple[xr.DataArray, xr.DataArray]:
    """Compute the Liao-style precipitation centroid latitude for each longitude."""
    field = precip.transpose(lat_name, lon_name).sortby(lat_name)
    band = field.where((field[lat_name] >= lat_min) & (field[lat_name] <= lat_max), drop=True)

    if band.sizes.get(lat_name, 0) == 0 or band.sizes.get(lon_name, 0) == 0:
        empty = xr.DataArray(
            np.full(field.sizes[lon_name], np.nan, dtype=float),
            coords={lon_name: field[lon_name].values},
            dims=lon_name,
            name='itcz_latitude',
        )
        counts = xr.DataArray(
            np.zeros(field.sizes[lon_name], dtype=int),
            coords={lon_name: field[lon_name].values},
            dims=lon_name,
            name='valid_cell_count',
        )
        return empty, counts

    lat_deg = np.asarray(band[lat_name].values, dtype=float)
    lat_rad = np.deg2rad(lat_deg)
    values = np.asarray(band.values, dtype=float)
    finite = np.isfinite(values)
    weights = np.where(finite, values, 0.0)
    coslat = np.cos(lat_rad)[:, None]

    numerator = np.trapezoid(lat_deg[:, None] * weights * coslat, x=lat_rad, axis=0)
    denominator = np.trapezoid(weights * coslat, x=lat_rad, axis=0)
    centroid = np.where(np.isfinite(denominator) & (denominator > 0.0), numerator / denominator, np.nan)
    valid_counts = finite.sum(axis=0).astype(int)

    line = xr.DataArray(
        centroid,
        coords={lon_name: band[lon_name].values},
        dims=lon_name,
        name='itcz_latitude',
    )
    counts = xr.DataArray(
        valid_counts,
        coords={lon_name: band[lon_name].values},
        dims=lon_name,
        name='valid_cell_count',
    )
    return line, counts


def plot_itcz_map(
    precip: xr.DataArray,
    nh_line: xr.DataArray,
    sh_line: xr.DataArray,
    full_line: xr.DataArray,
    title: str,
    levels: np.ndarray,
    contour_level: float = 5.0,
    extent: list[float] | tuple[float, float, float, float] = PLOT_EXTENT,
):
    fig = plt.figure(figsize=(12.5, 5.8))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

    cf = ax.contourf(
        precip['lon'],
        precip['lat'],
        precip,
        levels=levels,
        cmap='YlGnBu',
        extend='max',
        transform=ccrs.PlateCarree(),
    )
    contour = ax.contour(
        precip['lon'],
        precip['lat'],
        precip,
        levels=[contour_level],
        colors='black',
        linewidths=1.0,
        linestyles='--',
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(contour, fmt={contour_level: f'{contour_level:g} mm month$^{{-1}}$'}, fontsize=8)

    ax.plot(
        nh_line['lon'],
        nh_line,
        color='crimson',
        linewidth=4,
        label='NH centroid (0-20N)',
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    ax.plot(
        sh_line['lon'],
        sh_line,
        color='royalblue',
        linewidth=4,
        label='SH centroid (20S-0)',
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    ax.plot(
        full_line['lon'],
        full_line,
        color='black',
        linewidth=5.5,
        label='-20, 20 Centroid',
        transform=ccrs.PlateCarree(),
        zorder=5,
    )

    ax.coastlines(resolution='110m', linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_xticks(np.arange(extent[0], extent[1] + 1.0, 10.0), crs=ccrs.PlateCarree())
    ax.set_yticks(np.arange(extent[2], extent[3] + 1.0, 5.0), crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())

    ax.gridlines(draw_labels=False, linewidth=0.35, linestyle='--', color='gray', alpha=0.5)

    cbar = fig.colorbar(cf, ax=ax, pad=0.02, shrink=0.9)
    cbar.set_label('Rainfall climatology (mm month$^{-1}$)')

    ax.set_title(title, loc='left', fontweight='bold')
    ax.legend(loc='lower right', frameon=True)
    plt.tight_layout()
    plt.show()


def plot_itcz_lon_profile(
    raw_nh: xr.DataArray,
    raw_sh: xr.DataArray,
    raw_full: xr.DataArray,
    smooth_nh: xr.DataArray,
    smooth_sh: xr.DataArray,
    smooth_full: xr.DataArray,
    extent: list[float] | tuple[float, float, float, float] = PLOT_EXTENT,
):
    fig, ax = plt.subplots(figsize=(12.0, 4.8))
    ax.plot(raw_nh['lon'], raw_nh, color='crimson', linewidth=2.0, label='Raw NH')
    ax.plot(raw_sh['lon'], raw_sh, color='royalblue', linewidth=2.0, label='Raw SH')
    ax.plot(raw_full['lon'], raw_full, color='black', linewidth=4.0, label='Raw -20, 20 Centroid')
    ax.plot(smooth_nh['lon'], smooth_nh, color='crimson', linewidth=2.0, linestyle='--', label='Smoothed NH')
    ax.plot(smooth_sh['lon'], smooth_sh, color='royalblue', linewidth=2.0, linestyle='--', label='Smoothed SH')
    ax.plot(smooth_full['lon'], smooth_full, color='black', linewidth=4.0, linestyle='--', label='Smoothed -20, 20 Centroid')
    ax.axhline(0.0, color='0.35', linewidth=0.9)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('ITCZ latitude (degrees)')
    ax.set_title('ITCZ latitude versus longitude', loc='left', fontweight='bold')
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5)
    ax.legend(ncols=3, frameon=True)
    plt.tight_layout()
    plt.show()

In [ ]:
rain = xr.open_dataset(MSWEP_PATH, chunks={'time': 12})['precipitation']
rain = rain.assign_coords(lon=(rain.lon % 360)).sortby('lon')
rain = rain.sel(
    time=slice(f'{START_YEAR - 1}-12-01', f'{END_YEAR}-02-29'),
    lat=slice(25, -25),
    lon=slice(70, 290),
)
rain = rain.where(np.isfinite(rain) & (rain != -9999.0))

# Keep the native monthly rainfall accumulation units: mm month-1.
djf_climatology, complete_years = compute_djf_climatology(rain, START_YEAR, END_YEAR)

djf_climatology = djf_climatology.where(
    (djf_climatology['lat'] >= PLOT_EXTENT[2])
    & (djf_climatology['lat'] <= PLOT_EXTENT[3])
    & (djf_climatology['lon'] >= PLOT_EXTENT[0])
    & (djf_climatology['lon'] <= PLOT_EXTENT[1]),
    drop=True,
).load()

djf_smoothed = smooth_precip_gaussian(djf_climatology, sigma=SMOOTH_SIGMA).load()

raw_nh_line, raw_nh_count = compute_itcz_centroid(djf_climatology, lat_min=NH_BAND[0], lat_max=NH_BAND[1])
raw_sh_line, raw_sh_count = compute_itcz_centroid(djf_climatology, lat_min=SH_BAND[0], lat_max=SH_BAND[1])
raw_full_line, raw_full_count = compute_itcz_centroid(djf_climatology, lat_min=FULL_BAND[0], lat_max=FULL_BAND[1])
smoothed_nh_line, smoothed_nh_count = compute_itcz_centroid(djf_smoothed, lat_min=NH_BAND[0], lat_max=NH_BAND[1])
smoothed_sh_line, smoothed_sh_count = compute_itcz_centroid(djf_smoothed, lat_min=SH_BAND[0], lat_max=SH_BAND[1])
smoothed_full_line, smoothed_full_count = compute_itcz_centroid(djf_smoothed, lat_min=FULL_BAND[0], lat_max=FULL_BAND[1])

max_level = float(np.ceil(np.nanpercentile(np.asarray(djf_climatology.values, dtype=float), 99.0) / 20.0) * 20.0)
max_level = max(max_level, 120.0)
rain_levels = np.arange(0.0, max_level + 20.0, 20.0)
if rain_levels.size < 8:
    rain_levels = np.linspace(0.0, max_level, 9)

valid_count_table = pd.DataFrame({
    'lon': raw_nh_line['lon'].values,
    'raw_nh_valid_cells': raw_nh_count.values.astype(int),
    'raw_sh_valid_cells': raw_sh_count.values.astype(int),
    'raw_full_valid_cells': raw_full_count.values.astype(int),
    'smoothed_nh_valid_cells': smoothed_nh_count.values.astype(int),
    'smoothed_sh_valid_cells': smoothed_sh_count.values.astype(int),
    'smoothed_full_valid_cells': smoothed_full_count.values.astype(int),
})

print(f'Complete DJF seasons used: {int(complete_years.min())}-{int(complete_years.max())} ({len(complete_years)} seasons)')
print(f'Source rainfall units: {rain.attrs.get("units", "unknown")}')
print(f'Working rainfall units: {djf_climatology.attrs.get("units", "unknown")}')
print(f'Longitude range used: {float(djf_climatology.lon.min()):.2f}E to {float(djf_climatology.lon.max()):.2f}E')
print(f'Latitude range shown: {float(djf_climatology.lat.min()):.2f} to {float(djf_climatology.lat.max()):.2f}')
print(f'Latitude bands used: NH = {NH_BAND[0]:.0f} to {NH_BAND[1]:.0f} deg, SH = {SH_BAND[0]:.0f} to {SH_BAND[1]:.0f} deg, Full = {FULL_BAND[0]:.0f} to {FULL_BAND[1]:.0f} deg')
print(f'Raw climatology rainfall min/max: {float(djf_climatology.min(skipna=True)):.3f} to {float(djf_climatology.max(skipna=True)):.3f} mm month-1')
print(f'Smoothed climatology rainfall min/max: {float(djf_smoothed.min(skipna=True)):.3f} to {float(djf_smoothed.max(skipna=True)):.3f} mm month-1')
print('')
print('Valid grid cells per longitude used in the centroid calculation: first 12 longitudes')
print(valid_count_table.head(12).to_string(index=False))
print('')
print('Valid grid cells per longitude used in the centroid calculation: last 12 longitudes')
print(valid_count_table.tail(12).to_string(index=False))

In [ ]:
plot_itcz_map(
    djf_climatology,
    raw_nh_line,
    raw_sh_line,
    raw_full_line,
    title='Raw DJF rainfall climatology and Liao centroid ITCZ',
    levels=rain_levels,
    contour_level=RAIN_CONTOUR_LEVEL,
)

In [ ]:
plot_itcz_map(
    djf_smoothed,
    smoothed_nh_line,
    smoothed_sh_line,
    smoothed_full_line,
    title='Smoothed DJF rainfall climatology and Liao centroid ITCZ',
    levels=rain_levels,
    contour_level=RAIN_CONTOUR_LEVEL,
)

In [ ]:
plot_itcz_lon_profile(
    raw_nh_line,
    raw_sh_line,
    raw_full_line,
    smoothed_nh_line,
    smoothed_sh_line,
    smoothed_full_line,
)